# The SteelVault reference — what the graph of this contract should hold

The reference `evaluate_kg.ipynb` scores extractions against. It was written by reading the
whole contract, revised after an independent second reading, and every span in it is checked
to be verbatim before the file is written.

**This notebook is the source of the reference.** To change it, edit the tables below and run
it again: it rewrites `infra/json/gold/<contract>.json`, and it refuses to write a span that is
not word for word in the paragraphs it cites. Add a line to `revisions` saying what changed
and why — a reference that moves silently makes every earlier score meaningless.

In [1]:
import json
import re
import unicodedata
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == "KG" else Path.cwd()
STEM = "root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement"
paragraphs = json.loads((ROOT / "infra/json/paragraphs" / f"{STEM}.json").read_text())["paragraphs"]
by_short = {"p-" + p["id"].split("-p-")[-1]: p for p in paragraphs}

E, A, G = "equidata", "affiliate", "guarantor"

## Statements

`(id, paragraphs, kind, burden, benefit, text, flags)`. The first paragraph decides the
expected clause. Flags: `opt` — defensible either way, never a miss; `nonparty` — a duty of
someone outside the contract (the arbitrators); `burden-or-null` — the burden may also be the
guarantor party, which the extraction prompt leaves null.

`burden` is the party acted on: the obligor of a duty, the restrained party of a prohibition,
and for a right the party that must submit to it. `benefit` is the holder of a right, or the
party a duty is owed to.

In [2]:
S = [
    # Permission (unnumbered)
    ("S01", "p-12", "right", E, A, "Marketing Affiliate may display Marketing Materials at its principal place(s) of business, or at the principal place(s) of its third party partners, together with a link from the Marketing Affiliate Web Site to Fquidata (and its partners) Web Site", ""),
    ("S02", "p-12", "right", E, A, "Marketing Materials may also be used in the marketing of potential customers through direct mail and personal solicitation as well as inbound and outbound telemarketing", ""),
    ("S03", "p-12", "prohibition", A, E, "Marketing Affiliate may not otherwise offer for sale, market, sell or distribute the Services of Equidata without express written permission", ""),
    # 1. Compensation
    ("S04", "p-13", "obligation", A, E, "Marketing Affiliate shall be responsible for collecting all amounts due directly from the Consumer", ""),
    ("S05", "p-13", "obligation", A, E, "shall bear sole responsibility for non-payment of any fees charged to the Consumer", ""),
    ("S06", "p-13", "obligation", A, E, "Marketing Affiliate shall pay to Equidata, as compensation for its providing of Services under this agreement, such amounts as outlined and detailed in Exhibit A attached hereto", ""),
    ("S07", "p-13", "obligation", E, A, "Such amounts shall be billed on a bi-monthly basis by Equidata", "opt"),
    # A payment term is the deadline of the payment duty (S06), not an act of its own.
    ("S08", "p-13", "obligation", A, E, "are due and payable in full by Marketing Affiliate 30 days from the invoice date", "opt"),
    ("S09a", "p-13", "obligation", E, A, "Said amounts charged to Equidata will be billed separately to Marketing Affiliate", "opt"),
    ("S09", "p-13", "obligation", A, E, "are due immediately upon receipt", ""),
    ("S10", "p-13", "obligation", A, E, "Marketing Affiliate agrees to reimburse Equidata all costs of collecting any past due amounts from Marketing Affiliate by reason of non payment, including reasonable attorney fees and disbursements", ""),
    ("S11", "p-13", "right", A, E, "Equidata reserves the right to increase the base cost of Services", ""),
    ("S12", "p-13", "obligation", E, A, "Notice will be given to Marketing Affiliate in writing no less than 30 days prior to such increase taking affect", ""),
    ("S13", "p-13", "obligation", A, E, "A development fee, yet To Be Determined and outlined in Exhibit A, is due upon a signed agreement of project scope", ""),
    ("S14", "p-14", "obligation", A, E, "Marketing Affiliate agrees to pay promptly and in full all charges incurred through services rendered when billed", ""),
    ("S15", "p-14", "obligation", A, E, "When paying by credit card, the Marketing Affiliate agrees to pay for all items that are revoked or disputed by the credit card company or the card holder along with any charges or fees charged by the credit card company including fees associated with processing the credit card transaction and that the Marketing Affiliate will be billed for those items in accordance with Equidata standard practices", ""),
    # The author's decision (2026-10-05): the personal guarantor stays null, as the prompt
    # says for a role the contract does not name; a guarantor party is still accepted.
    ("S16", "p-14", "obligation", None, E, "this Agreement will serve as a personal guaranty by the undersigned principal, partner or owner of the company, and the undersigned principal, partner or owner will become responsible for any unpaid balance past due on any invoice", "burden-or-null"),
    ("S17", "p-14", "obligation", A, E, "The Marketing Affiliate agrees to pay a late charge of 1 l/2% per month on the unpaid, past-due amount", ""),
    ("S18", "p-14", "obligation", A, E, "as well as a returned check fee of not less than $35.00 per returned item", ""),
    ("S19", "p-14", "obligation", A, E, "the Marketing Affiliate agrees to pay 25% attorney’s fees plus court cost in the event that the Marketing Affiliate’s account is referred to an attorney for collection", ""),
    # 2. Disputes
    ("S20a", "p-15", "right", A, E, "Equidata or Marketing Affiliate may choose arbitration", ""),
    ("S20b", "p-15", "right", E, A, "Equidata or Marketing Affiliate may choose arbitration", ""),
    ("S21a", "p-15", "obligation", A, E, "Marketing Affiliate and Equidata shall be obligated by the terms agreed upon by arbitration", ""),
    ("S21b", "p-15", "obligation", E, A, "Marketing Affiliate and Equidata shall be obligated by the terms agreed upon by arbitration", ""),
    ("S22", "p-15", "obligation", A, E, "all monies determined owed shall be considered due and payable immediately", "opt"),
    ("S23", "p-15", "obligation", A, E, "Such arbitration does not relieve Marketing Affiliate from its obligation to promptly pay for undisputed charges in accordance with the terms of this Agreement", ""),
    ("S24a", "p-15", "obligation", A, E, "Such disputes shall be settled by arbitration in the City of Newport News, Virginia", "opt"),
    ("S24b", "p-15", "obligation", E, A, "Such disputes shall be settled by arbitration in the City of Newport News, Virginia", "opt"),
    ("S25", "p-16", "obligation", A, E, "Marketing Affiliate shall give Equidata written demand of dispute within 10 days of the due date of the invoice", ""),
    ("S26", "p-16", "obligation", A, E, "The demand shall set forth a statement for the nature of the dispute and the amount involved", "opt"),
    ("S27a", "p-16", "obligation", A, E, "the parties shall jointly select an arbitrator", ""),
    ("S27b", "p-16", "obligation", E, A, "the parties shall jointly select an arbitrator", ""),
    ("S28a", "p-20", "obligation", A, E, "each party will select an arbitrator of their choosing", ""),
    ("S28b", "p-20", "obligation", E, A, "each party will select an arbitrator of their choosing", ""),
    ("N1", "p-20", "obligation", None, None, "the two arbitrators will jointly select a third arbitrator(s)", "nonparty"),
    ("N2", "p-20", "obligation", None, None, "the arbitrator(s) shall schedule the arbitration hearing to commence on a mutually convenient date", "nonparty"),
    ("N3", "p-20", "obligation", None, None, "The hearing shall commence no later than 25 calendar days after Equidata receives receipt of dispute from Marketing Affiliate and shall continue from day to day until completed", "nonparty"),
    ("N4", "p-20", "obligation", None, None, "The arbitrator(s) shall issue an award in writing no later than 10 calendar days after the conclusion of the hearing", "nonparty"),
    ("S29a", "p-20", "obligation", A, E, "The arbitration award shall be final and binding on both parties", "opt"),
    ("S29b", "p-20", "obligation", E, A, "The arbitration award shall be final and binding on both parties", "opt"),
    # 3. Operational Specifications
    ("S30a", "p-21", "obligation", A, E, "Marketing Affiliate and Equidata shall agree upon Operational Specifications pertaining to the methodology and logistics of data transfer and database coordination", ""),
    ("S30b", "p-21", "obligation", E, A, "Marketing Affiliate and Equidata shall agree upon Operational Specifications pertaining to the methodology and logistics of data transfer and database coordination", ""),
    ("S31a", "p-21", "obligation", A, E, "Both parties must agree upon any changes to the Operational Specifications in writing", "opt"),
    ("S31b", "p-21", "obligation", E, A, "Both parties must agree upon any changes to the Operational Specifications in writing", "opt"),
    # 4. Non-solicitation
    ("S32", "p-22", "prohibition", A, E, "Marketing Affiliate shall not directly or indirectly solicit an existing business customer of Equidata during the term and condition of this Agreement other than for joint marketing purposes", ""),
    ("S33", "p-22", "prohibition", A, E, "Marketing Affiliate shall not market similar products from competing companies on any Web Site Landing Page containing the Equidata or Marketing Affiliate Web link as long as this Agreement is in effect", ""),
    # 5. Compliance — the lead-in binds both parties, each item is one prohibition per party
    ("S34a", "p-23", "prohibition", A, E, "Marketing Affiliate nor Equidata, shall engage in any practice or activity that is not in compliance with the Fair Credit Reporting Act (FCRA), the Fair Debt Collection Practices Act (FDCPA) and the Health Insurance Portability and Accountability Act (HIPAA)", ""),
    ("S34b", "p-23", "prohibition", E, A, "Marketing Affiliate nor Equidata, shall engage in any practice or activity that is not in compliance with the Fair Credit Reporting Act (FCRA), the Fair Debt Collection Practices Act (FDCPA) and the Health Insurance Portability and Accountability Act (HIPAA)", ""),
    ("S35a", "p-24", "prohibition", A, E, "Violates any applicable law or regulation; including but not limited to the sale of illegal goods or the violation of export control or obscenity laws; that invade the privacy of any third party; that are in any way connected with the transmission of “junk mail”, “spam” or the unsolicited mass distribution of e-mail, or with any unethical marketing practices", ""),
    ("S35b", "p-24", "prohibition", E, A, "Violates any applicable law or regulation; including but not limited to the sale of illegal goods or the violation of export control or obscenity laws; that invade the privacy of any third party; that are in any way connected with the transmission of “junk mail”, “spam” or the unsolicited mass distribution of e-mail, or with any unethical marketing practices", ""),
    ("S36a", "p-25", "prohibition", A, E, "Is misleading, deceptive, confusing or abusive as outlined in the Telemarketing Fraud Prevention Act", ""),
    ("S36b", "p-25", "prohibition", E, A, "Is misleading, deceptive, confusing or abusive as outlined in the Telemarketing Fraud Prevention Act", ""),
    ("S37a", "p-26", "prohibition", A, E, "Makes any representation or statement, or grants any warranty or creates any other obligation with respect to the Services, that is in addition to or otherwise inconsistent with any representation, statement or warranty stated expressly by Equidata", ""),
    ("S37b", "p-26", "prohibition", E, A, "Makes any representation or statement, or grants any warranty or creates any other obligation with respect to the Services, that is in addition to or otherwise inconsistent with any representation, statement or warranty stated expressly by Equidata", "opt"),
    ("S38a", "p-27", "prohibition", A, E, "Uses Marketing Materials, media or methods that are not approved, including, but nor limited to telemarketing scripts", ""),
    ("S38b", "p-27", "prohibition", E, A, "Uses Marketing Materials, media or methods that are not approved, including, but nor limited to telemarketing scripts", "opt"),
    ("S39a", "p-27", "prohibition", E, A, "Such approval shall not be unreasonably withheld", ""),
    ("S39b", "p-27", "obligation", E, A, "shall be completed within 48 hours of receipt of Marketing Materials for review", ""),
    ("S40a", "p-28", "prohibition", A, E, "Does not meet the standards for good industry practices for the direct marketing industry", ""),
    ("S40b", "p-28", "prohibition", E, A, "Does not meet the standards for good industry practices for the direct marketing industry", "opt"),
    # 6. Audit
    ("S41", "p-30", "right", A, E, "Equidata may audit, at Equidata’s expense, the Marketing Affiliate’s marketing, practices and activities for the purpose of assuring compliance with this Agreement", ""),
    # The cost of the audit is Equidata's: an allocation in the Affiliate's favour.
    ("S41c", "p-30", "obligation", E, A, "Equidata may audit, at Equidata’s expense, the Marketing Affiliate’s marketing, practices and activities for the purpose of assuring compliance with this Agreement", "opt"),
    ("S42", "p-30", "right", A, E, "Equidata reserves the right to site inspect Marketing Affiliate’s physical location of business at any time", ""),
    # 7. Term and Termination
    ("S43a", "p-33", "right", A, E, "This Agreement may be terminated by either party with cause upon thirty (30) days written notice", ""),
    ("S43b", "p-33", "right", E, A, "This Agreement may be terminated by either party with cause upon thirty (30) days written notice", ""),
    ("S44", "p-33", "right", A, E, "Upon Marketing Affiliate’s default in payment or other breach of this Agreement, Equidata may terminate this Agreement without notice to Marketing Affiliate", ""),
    ("S45", "p-33", "right", A, E, "Upon termination for any reason, Equidata reserves the right to deactivate Marketing Affiliate’s access to the services including the Equidata Web Site", ""),
    ("S46", "p-33", "obligation", A, E, "Termination does not release Marketing Affiliate from paying all amounts owed to Equidata", ""),
    ("S47", "p-34", "obligation", A, E, "At time of Agreement termination, Marketing Affiliate shall immediately remove all URL related data pertaining to said Agreement", ""),
    ("S48", "p-34", "right", A, E, "if data is not voluntarily removed, Equidata reserves the right to use all available legal resources to force the removal of Equidata URL related data", ""),
    ("S49", "p-34", "obligation", A, E, "Marketing Affiliate agrees to be liable for the cost of such action, including but not limited to reasonable attorney fees", ""),
    ("S50", "p-35", "right", A, E, "Equidata reserves the right to terminate this Agreement immediately for cause if Experian, Equifax and/or TransUnion (Credit Reporting Agencies — CRAs) decline to render Services to Marketing Affiliate for any reason or if Equidata is notified by any of the CRAs to cease rendering Services to Marketing Affiliate", ""),
    # 8. Representations and Warranties — obligations of the party giving them (decision of
    # 2026-10-05: the ontology stays as it is, warranties are mapped onto Obligation)
    ("W1", "p-37 p-36", "obligation", A, E, "Marketing Affiliate does not engage in any business with respect to, and the Marketing Affiliate Web Site will not be used, or display any materials, in any form or medium, in connection with a credit clinic, credit repair or restoration, credit counseling firm, financial counseling firm, detective agency, private investigation, security services, practice of law, news reporting or journalism, or fraudulent or unethical conduct", ""),
    ("W2", "p-38 p-36", "obligation", A, E, "The information regarding Marketing Affiliate set forth in this Agreement, and the information provided to Equidata with respect to Marketing Affiliate and the Marketing Affiliate Web Site, is accurate", ""),
    ("W3", "p-39 p-41 p-36", "obligation", A, E, "Marketing Affiliate’s business, including without limitation any business conducted in connection with the Marketing Affiliate Web Site, does not violate any applicable law, regulation, court order or material agreement to which Marketing Affiliate is subject", ""),
    # 8.1 also promises what the web site will not do: a covenant inside the warranty.
    ("W1p", "p-37", "prohibition", A, E, "the Marketing Affiliate Web Site will not be used, or display any materials, in any form or medium, in connection with a credit clinic, credit repair or restoration, credit counseling firm, financial counseling firm, detective agency, private investigation, security services, practice of law, news reporting or journalism, or fraudulent or unethical conduct", "opt"),
    ("W4", "p-42", "obligation", E, A, "Equidata warrants that it is an authorized provider of the Services as outlined in this Agreement and that it has the ability to provide said Services in the manner described herein", ""),
    # 9. Indemnification (mutual)
    ("S51a", "p-46", "obligation", E, A, "Equidata and Marketing Affiliate each hereby agree to defend, indemnify and hold harmless each other and each of its employees, agents, officers, directors and shareholders from and against any claims, suits, demand or actions arising from breach of any warranties under this Agreement or failure to provide Services under this Agreement", ""),
    ("S51b", "p-46", "obligation", A, E, "Equidata and Marketing Affiliate each hereby agree to defend, indemnify and hold harmless each other and each of its employees, agents, officers, directors and shareholders from and against any claims, suits, demand or actions arising from breach of any warranties under this Agreement or failure to provide Services under this Agreement", ""),
    # 10. Proprietary Information — "Receiving Party" is each party in turn
    ("S52a", "p-47", "prohibition", A, E, "The Receiving Party may not disclose or use the Disclosing Party’s Confidential and Proprietary Information for any reason other than in the performance of this Agreement", ""),
    ("S52b", "p-47", "prohibition", E, A, "The Receiving Party may not disclose or use the Disclosing Party’s Confidential and Proprietary Information for any reason other than in the performance of this Agreement", ""),
    ("S53", "p-47", "prohibition", E, A, "any information received or collected by Marketing Affiliate about its Customers or potential Customers, including information used to enroll Customers is Proprietary as defined by this section and will not be used by Equidata in any manner other than as outlined herein", ""),
    # 11. Liability — disclaimers and exclusions are freedoms of Equidata, the one-sided indemnity a duty of the Affiliate
    ("S54", "p-48", "right", A, E, "MARKETING AFFILIATE ACKNOWLEDGES AND AGREES THAT ANY PRODUCT, SERVICE, LICENSE OR PERMISSION PROVIDED BY EQUIDATA UNDER THIS AGREEMENT IS PROVIDED ON AN “AS IS” BASIS", "opt"),
    ("S55", "p-48", "right", A, E, "EQUIDATA EXPRESSLY DISCLAIMS ANY WARRANTY OF ANY KIND WHATSOEVER, WHETHER EXPRESS, IMPLIED, STATUTORY, OR ARISING FROM COURSE OF DEALING OR PERFORMANCE, AND HEREBY DISCLAIMS AND EXCLUDES FROM THIS AGREEMENT ALL IMPLIED WARRANTIES", ""),
    ("S56", "p-48", "right", A, E, "IN NO EVENT WILL EQUIDATA BE LIABLE FOR ANY INDIRECT, EXEMPLARY, PUNITIVE, SPECIAL, OR CONSEQUENTIAL DAMAGES INCLUDING WITHOUT LIMITATION LOST PROFITS OR OTHER ECONOMIC LOSS, LOST REIMBURSEMENTS, AND LOST DATA, OR FOR ANY CLAIM BY ANY THIRD PARTY", ""),
    ("S57", "p-48", "obligation", A, E, "MARKETING AFFILIATE AGREES TO DEFEND, INDEMNIFY AND HOLD HARMLESS EQUIDATA, AND EACH OF ITS EMPLOYEES, AGENTS, OFFICERS AND DIRECTORS, FROM AND AGAINST ANY CLAIM, SUIT, DEMAND, OR ACTION, INCLUDING WITHOUT LIMITATION ATTORNEY FEES, ARISING FROM (A) BREACH OF THIS AGREEMENT BY MARKETING AFFILIATE, (B) THE MARKETING AFFILIATE WEB SITE, OR (C) MARKETING AFFILIATE’S BUSINESS", ""),
    # 12. Miscellaneous — only the parts that bind a party
    ("S58a", "p-49", "obligation", A, E, "All notices required to be given in writing must be sent by overnight delivery service to the name and address designated in this Agreement", ""),
    ("S58b", "p-49", "obligation", E, A, "All notices required to be given in writing must be sent by overnight delivery service to the name and address designated in this Agreement", ""),
    ("S59a", "p-49", "right", E, A, "to such other address that the receiving party may in advance designate by written notice", "opt"),
    ("S59b", "p-49", "right", A, E, "to such other address that the receiving party may in advance designate by written notice", "opt"),
    ("S60a", "p-49", "obligation", A, E, "Each party shall be solely responsible for supervision, daily direction and control of its employees and payment of their salaries, worker’s compensation, disability and other benefits", ""),
    ("S60b", "p-49", "obligation", E, A, "Each party shall be solely responsible for supervision, daily direction and control of its employees and payment of their salaries, worker’s compensation, disability and other benefits", ""),
    ("S61a", "p-49", "right", E, A, "shall have exclusive control of the manner and means of performing such obligations", "opt"),
    ("S61b", "p-49", "right", A, E, "shall have exclusive control of the manner and means of performing such obligations", "opt"),
]

## Text that states no duty, right or prohibition of a party

A statement the graph builds from one of these spans is counted as noise.

In [3]:
NON_DEONTIC = [
    ("p-5", "preamble", "This Agreement is made this 1st day of October 2008"),
    ("p-7", "recital", "Equidata provides certain personal credit, fraud detection, credit scoring services and credit monitoring for consumers"),
    ("p-8", "recital", "Marketing Affiliate and Equidata wish to enter into an agreement under which Marketing Affiliate may market the Services"),
    ("p-9", "recital", "Marketing Affiliate wishes to market the Services indirectly through third party programs, direct mail, Internet and both inbound and outbound telemarketing. In addition, each may own and operate a web site utilizing direct access to the Services through Internet links"),
    ("p-10", "recital", "Therefore, if accepted all parties agree that the following shall constitute a marketing agreement between the parties"),
    ("p-13", "price scope", "The prices set forth in Exhibit A do not include regulatory fees, sales tax, excise tax or any other fees or taxes that may be charged by states or local taxing authorities nor does it include additional fees or surcharges, including specific area Affiliate charges that may be accessed by the Credit Reporting Agencies (CRA’s)"),
    ("p-15", "definition", "In the case of disputed charge, defined as a non-payment of an invoice for which notice of dispute has been given in writing by Marketing Affiliate to Equidata"),
    ("p-21", "deeming", "Upon mutual agreement as to the Operational Specifications, they shall be deemed to be a part of this Agreement by way of an Exhibit"),
    ("p-21", "deeming", "Any such changes will be deemed to be a part of the Operational Specifications"),
    ("p-29", "pointer", "Further guidelines and requirements are provided in Exhibit B and C"),
    ("p-31", "term", "This Agreement commences on the Effective Date, and terminates, along with all licenses and authorizations granted under it, upon the earliest of termination in accordance with the following"),
    ("p-32", "term and renewal", "This Agreement shall be for the term of one year; thereafter, the Agreement shall renew automatically under these same terms and agreements unless superceded by future agreements"),
    ("p-47", "acknowledgement", "Marketing Affiliate and Equidata mutually acknowledge that from time to time Confidential Information may be received by each"),
    ("p-47", "definition", "Confidential Information, includes, but is not limited to, Customer names and lists"),
    ("p-49", "successors", "This Agreement binds and inures to the benefit of each party’s permitted successors, assigns and legal representatives, including the purchasers of the stock or assets of either party hereto"),
    ("p-49", "deeming", "No delegation by either party of any duty hereunder shall be deemed an assignment of this Agreement, nor shall any change in control or an assignment of by operation of law by either party be deemed an assignment hereunder"),
    ("p-49", "no waiver", "Any failure or delay in exercising, or any single or partial exercise of, any right or remedy by either party may not be deemed a waiver of any further, prior, or future right or remedy hereunder"),
    ("p-49", "governing law", "This Agreement is governed by and construed in accordance with the laws of the State of Virginia"),
    ("p-49", "deeming", "Notice is deemed effective on the day after delivery by the overnight carrier"),
    ("p-49", "severability", "If any provision of this Agreement is declared invalid, the other provisions remain in full force and effect and this Agreement is deemed to be amended to replace, to the extent legally possible, the rights and obligations contained in the invalid provision"),
    ("p-49", "severability", "The invalidity of any provision is not a failure of consideration"),
    ("p-49", "status", "The Parties shall operate as Independent Contractors in performing their obligations under the Agreement"),
    ("p-49", "status", "Nothing in the Agreement shall be construed as making either party the agent of the other party, as granting to the other party the right to enter into any contract on behalf of the other party, or as establishing an association, franchise, joint venture or partnership between the Parties"),
    ("p-49", "status", "Under no circumstances shall the employees of one party be deemed to be employees of the other for any purpose"),
    ("p-49", "entire agreement", "This Agreement constitutes the entire understanding of the parties with respect to the subject matter hereof, and supersedes all prior or contemporaneous agreements, statements and representations, oral or written, between the parties relating to the subject matter of the Agreement"),
    ("p-49", "formality", "No representation or promise, or modification or amendment to this Agreement is binding on either party unless in writing signed by authorized representatives of both parties"),
]

## Conditions and values

A condition gates one provision; for a bilateral pair it points at the first twin and counts
for both. A value lists the spellings it may come in, and what it quantifies — a statement, or
`clause:N`.

In [4]:
# (id, paragraph, operator, trigger, gates, flags) — one per provision; a bilateral pair is one provision
C = [
    ("C01", "p-12", "UNLESS", "express written permission", "S03", "opt"),
    ("C02", "p-13", "IF", "by reason of non payment", "S10", ""),
    ("C03", "p-13", "UPON", "upon a signed agreement of project scope", "S13", ""),
    ("C04", "p-14", "IF", "When paying by credit card", "S15", ""),
    ("C05", "p-14", "IF", "in the event that the Marketing Affiliate’s account is referred to an attorney for collection", "S19", ""),
    ("C06", "p-14", "IF", "on the unpaid, past-due amount", "S17", ""),
    ("C07", "p-15", "IF", "In the case of disputed charge", "S20a", ""),
    ("C08", "p-16", "IF", "If Equidata and Marketing Affiliate can not resolve the dispute on their own within 10 days after Equidata receive said dispute", "S27a", ""),
    ("C09", "p-20", "IF", "If the parties do not agree on the selection of an arbitrator", "S28a", ""),
    ("C10", "p-33", "UPON", "Upon Marketing Affiliate’s default in payment or other breach of this Agreement", "S44", ""),
    ("C11", "p-33", "UPON", "Upon termination for any reason", "S45", ""),
    ("C12", "p-34", "UPON", "At time of Agreement termination", "S47", ""),
    ("C13", "p-34", "IF", "if data is not voluntarily removed", "S48", ""),
    ("C14", "p-35", "IF", "if Experian, Equifax and/or TransUnion (Credit Reporting Agencies — CRAs) decline to render Services to Marketing Affiliate for any reason", "S50", ""),
    ("C15", "p-35", "IF", "if Equidata is notified by any of the CRAs to cease rendering Services to Marketing Affiliate", "S50", ""),
    ("C16", "p-46", "IF", "arising from breach of any warranties under this Agreement or failure to provide Services under this Agreement", "S51a", ""),
    ("C17", "p-48", "IF", "ARISING FROM (A) BREACH OF THIS AGREEMENT BY MARKETING AFFILIATE", "S57", ""),
    ("C18", "p-33", "IF", "with cause", "S43a", "opt"),
    ("C19", "p-34", "IF", "if data is not voluntarily removed", "S49", "opt"),
    ("C20", "p-14", "IF", "any unpaid balance past due on any invoice", "S16", ""),
]

In [5]:
# (id, paragraph, accepted spellings, quantifies, flags) — "clause:7.1" points at a clause, not a statement
V = [
    ("V01", "p-13", ["30"], "S06", ""),
    ("V02", "p-13", ["30"], "S12", ""),
    ("V03", "p-14", ["1 1/2", "1 l/2", "1.5"], "S17", ""),
    ("V04", "p-14", ["35"], "S18", ""),
    ("V05", "p-14", ["25"], "S19", ""),
    ("V06", "p-16", ["10"], "S25", ""),
    ("V07", "p-16", ["10"], "S27a", "opt"),
    ("V08", "p-27", ["48"], "S39b", ""),
    ("V09", "p-32", ["one year", "one years", "1 year", "1 years", "one (1) year", "12 months"], "clause:7.1", ""),
    ("V10", "p-33", ["30", "thirty"], "S43a", ""),
    ("V11", "p-20", ["5"], "N2", "opt"),
    ("V12", "p-20", ["25"], "N3", "opt"),
    ("V13", "p-20", ["10"], "N4", "opt"),
]

## Parties

In [6]:
PARTIES = [
    {"key": E, "name": "Equidata, Inc.", "aliases": ["Equidata"]},
    {"key": A, "name": "National Credit Report.com, LLC", "aliases": ["Marketing Affiliate", "National Credit Report LLC", "National Credit Report.com"]},
    {
        "key": G,
        "name": "the undersigned principal, partner or owner",
        "aliases": ["undersigned principal", "undersigned principal, partner or owner", "Ivan Posniak"],
        "note": "Personal guarantor of the Marketing Affiliate's unpaid balance (§1). The contract never names them; 'Ivan Posniak' is inferred from the signature block, where he signs for the Marketing Affiliate as CEO.",
    },
]

## Check and write

Every span is checked against its paragraphs before anything is written.

In [7]:
def squash(text):
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip().lower()


def check(span, pids):
    pool = squash(" ".join(by_short[p]["text"] for p in pids))
    assert squash(span) in pool, f"not verbatim in {pids}: {span[:80]}"


statements = []
for sid, pids, kind, burden, benefit, text, flags in S:
    pids = pids.split()
    check(text, pids)
    statements.append(
        {
            "id": sid,
            "paragraphs": pids,
            "kind": kind,
            "burden": burden,
            "benefit": benefit,
            "text": text,
            "optional": flags == "opt",
            "nonParty": flags == "nonparty",
            **({"burdenAlso": [G]} if flags == "burden-or-null" else {}),
        }
    )
ids = {s["id"] for s in statements}

non_deontic = []
for pid, why, text in NON_DEONTIC:
    check(text, [pid])
    non_deontic.append({"paragraphs": [pid], "why": why, "text": text})

conditions = []
for cid, pid, op, trigger, gates, flags in C:
    check(trigger, [pid])
    assert gates in ids, gates
    conditions.append({"id": cid, "paragraphs": [pid], "operator": op, "trigger": trigger, "gates": gates, "optional": flags == "opt"})

values = []
for vid, pid, accept, quantifies, flags in V:
    assert quantifies in ids or quantifies.startswith("clause:"), quantifies
    values.append({"id": vid, "paragraphs": [pid], "accept": accept, "quantifies": quantifies, "optional": flags == "opt"})

## Conventions and revisions

The rules the tables above follow, and the history of every change to them. They travel
inside the file, so whoever reads a score can read what it was scored against.

In [8]:
gold = {
    "documentId": STEM,
    "status": "draft 2.3 — Claude's reading, revised after an independent second reading of the whole contract; pending review by the author",
    "conventions": [
        "One statement per distinct act. A new statement starts with a new modality (must / may / must not), a new acting party, or a trigger or amount of its own; a deadline or frequency is a field of the act it times, not an act; coordinated verbs or objects that share all three are one statement ('defend, indemnify and hold harmless'; 'fees plus court cost').",
        "A provision binding both parties ('each party', 'either party', 'both parties', 'X nor Y', 'the Receiving Party' in a mutual clause) is one statement per party, with the same text and the other party as benefit — as the extraction prompt instructs.",
        "burden is the party acted on: the obligor of a duty, the restrained party of a prohibition, and for a right the party that must submit to it. benefit is the holder of a right or the party a duty is owed to.",
        "Disclaimers and exclusions of liability are rights (freedoms) of the party they protect; the party that gives up the claim carries the burden.",
        "A representation or warranty is an obligation of the party giving it, owed to the other party: the ontology is not extended, warranties are mapped onto Obligation (decision of 2026-10-05). A forward-looking promise inside a warranty is also an optional prohibition.",
        "optional: a defensible statement whose absence is not counted as a miss, and whose kind and parties are not scored. nonParty: a duty of someone outside the contract (the arbitrators); neither a hit nor a miss. burdenAlso: other burdens accepted as right.",
        "A condition gates one provision. For a bilateral pair it points at the first twin and counts for both.",
        "The expected clause of each statement is the clause of its first paragraph in the paragraph file's tree, which was checked by reading against the contract; p-12 (Permission) is unnumbered. A warranty also lists p-36, the lead-in that carries its subject.",
        "Every span is verbatim in its paragraphs. W3 runs across a page break (p-39 into p-41) and is verbatim across the two.",
        "Signature blocks, page footers and initials carry no statements.",
    ],
    "revisions": [
        "Draft 2.3: S08 made optional and V01 moved to S06 — '30 days from the invoice date' is the deadline of the payment duty, which the ontology holds in its deadline field; convention 1 already said a deadline does not start a new statement.",
        "Draft 2.2: the ontology stays as it is; W1–W4 are obligations of the party warranting and count in the core recall.",
        "Draft 2.1: the author decided the personal guarantor of §1 stays null, as the prompt says for a role the contract does not name — S16's burden is null, a 'guarantor' party is still accepted.",
        "Draft 2 (second reading): S60a/b benefit set to the other party; S22 burden set to the Affiliate (a disputed charge is its non-payment); S09 split into billing (Equidata, optional) and payment (Affiliate); S39 split into the no-unreasonable-refusal prohibition and the 48-hour duty; S35a/b extended to the whole of 5.1; S15 extended to the billing practice; S31a/b and S40b made optional; added optional S41c (audit at Equidata's expense), W1p (the covenant in 8.1) and S61a/b (exclusive control of the means); S16 also accepts a null burden, because the prompt names this role as one to leave null; bilateral conditions reduced to one per provision; C06, C16, C17 made required; added C19 (S49) and C20 (the guaranty's trigger); C01 trigger no longer a double negative; nonDeontic 'status' reduced to the independent-contractor sentence; 'price scope' extended to the end of its sentence.",
    ],
    "parties": PARTIES,
    "statements": statements,
    "nonDeontic": non_deontic,
    "conditions": conditions,
    "values": values,
}

out = ROOT / "infra/json/gold" / f"{STEM}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(gold, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

core = [s for s in statements if not s["optional"] and not s["nonParty"]]
print("written", out.relative_to(ROOT))
print("statements", len(statements), "| core", len(core), "| optional", sum(s["optional"] for s in statements),
      "| non-party", sum(s["nonParty"] for s in statements))
print("non-deontic spans", len(non_deontic), "| conditions", len(conditions), "| values", len(values))

written infra/json/gold/root_SteelVaultCorp_20081224_10-K_EX-10_16_3074935_EX-10_16_Affiliate_Agreement.json
statements 94 | core 69 | optional 21 | non-party 4
non-deontic spans 26 | conditions 20 | values 13
